# To perform and visualize analyses online 

## Before the experiment: get everything ready 

### imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import hydra
from hydra.utils import get_original_cwd
import os
from omegaconf import DictConfig, OmegaConf
from dataclasses import dataclass
from typing import List, Dict, Any

from IPython.display import display



In [ ]:
# Load config
import sys
import os
from pathlib import Path


# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

# Import Hydra config utilities
from omegaconf import DictConfig, OmegaConf
import hydra
from hydra.utils import instantiate
from hydra.core.config_store import ConfigStore
from hydra import compose, initialize

# Initialize Hydra with the relative path to the config directory
config_path = os.path.join(relative_repo_path,"config")
print(f"Config path: {config_path}")

# Initialize Hydra
with initialize(version_base="1.3", config_path=config_path):
    # Compose the configuration
    cfg = compose(config_name="config")

# Print the config to verify it loaded correctly
print("Configuration loaded successfully:")
print(OmegaConf.to_yaml(cfg))



In [ ]:
from simulations.loop_components.dj_wrappers import DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper
from simulations.loop_components.recording_file_copier import copy_rec_files,create_directory_structure
from simulations.loop_components.stimulus import create_rf_avi_from_roi_ids


### Create processing components (connect them to DB)

In [ ]:
# create preprocessor
os.environ["DJ_SUPPORT_FILEPATH_MANAGEMENT"] = "TRUE"

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore
                data_subfolders=cfg.data_subfolders, # type: ignore


                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore

                    )



In [ ]:

# Load config and tables
dj_table_holder.load_config()
dj_table_holder.load_tables()
print(" loaded and configured successfully")
dj_table_holder.clear_tables("experiment")

# dj_table_holder.setup()




In [ ]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)


random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    model_configs=cfg.model_configs, 
    seeds= [123,456,789])

## During the experimet

### Move files from server to the repo 

In [ ]:
create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,
                           date=  cfg.data_subfolders.day, 
                           experiment = cfg.data_subfolders.experiment)

copy_rec_files(
    recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
    destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
    date=cfg.data_subfolders.day,  # type: ignore
    experiment=cfg.data_subfolders.experiment,  # type: ignore
    full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
)

In [ ]:
preprocessor.upload_iteration_metadata()

### Analysis and Visualization

In [ ]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
field_key = missing_keys[0]
missing_keys
# import datetime
# test_key = {'experimenter': 'closedlooptest',
#   'date': datetime.date(2025, 7, 17),
#   'exp_num': 1,
#   'raw_id': 1,
#   'field': 'GCL1',
#   'region': 'RR',
#   'cond1': 'iter0'}
# missing_keys = [test_key]

In [ ]:
# somehow I get a circular import error if I import this at the top
from simulations.gui.integrated_autorois import InteractiveRoiCanvas

In [ ]:
online_analysis_gui = InteractiveRoiCanvas(
    dj_table_holder=dj_table_holder,
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper,random_seed_mei_wrapper],
    field_key=field_key,
    canvas_width=30,
    )

In [ ]:
display(online_analysis_gui.start_gui()) 

# The stimulus

In [ ]:
create_rf_avi_from_roi_ids(roi_ids=[42,19],
                           peak_sta_position_table= dj_table_holder("PeakSTAPosition"),
                           abs_save_dir=os.path.join(cfg.paths.repo_directory,"stimuli","updated_loop_data"))